# HIPS Two-Problem Decomposition: Baseline + Sensitivity

The HIPS vs FTIR EC regression for Addis Ababa (slope=0.40, intercept=+2.84) reveals
**two separate problems**:

1. **Additive baseline** (~28 Mm⁻¹ in Fabs always present) → drives the intercept
2. **Compressed sensitivity** (HIPS captures only 40% of EC variability) → drives the low slope

Subtracting the baseline fixes the intercept but **does not** fix the slope.

| Section | Analysis |
|---------|----------|
| **1** | Cross-site comparison — is this Addis-specific? |
| **2** | Baseline subtraction — fix problem 1, expose problem 2 |
| **3** | Filter loading / saturation hypothesis — why is the slope 0.40? |
| **4** | Two-parameter correction — fix both problems |
| **5** | Validation — does the corrected HIPS agree better with Aethalometer? |

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'pyproject.toml').exists()), cwd)
data_root = Path(
    os.environ.get('AETHMODULAR_DATA_ROOT', str(repo_root / 'research' / 'ftir_hips_chem'))
).expanduser().resolve()

scripts_path = str((repo_root / 'research' / 'ftir_hips_chem' / 'scripts').resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE, FILTER_CATEGORIES
from data_matching import (
    load_aethalometer_data, load_filter_data, add_base_filter_id,
    match_all_parameters, pivot_filter_by_id,
)

FONT = {'title': 16, 'axis': 14, 'tick': 12, 'legend': 11, 'annot': 11}
plt.rcParams.update({
    'font.size': FONT['tick'], 'axes.titlesize': FONT['title'],
    'axes.labelsize': FONT['axis'], 'legend.fontsize': FONT['legend'],
})

SITE_COLORS = {name: cfg['color'] for name, cfg in SITES.items()}
SITE_MARKERS = {'Beijing': 'o', 'Delhi': 's', 'JPL': '^', 'Addis_Ababa': 'D'}

output_root = repo_root / 'artifacts' / 'notebook_outputs' / 'hips_two_problems'
plots_dir = output_root / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)
print(f'Repo root: {repo_root}')
print(f'MAC_VALUE: {MAC_VALUE}')

In [ ]:
# === Load all sites ===

filter_data = load_filter_data()
filter_data = add_base_filter_id(filter_data)
aethalometer_data = load_aethalometer_data()

site_data = {}
for site_name, site_cfg in SITES.items():
    code = site_cfg['code']
    df_aeth = aethalometer_data.get(site_name)
    bc = match_all_parameters(site_name, code, df_aeth, filter_data)
    if bc is not None:
        valid = bc.dropna(subset=['hips_fabs', 'ftir_ec']).copy()
        valid = valid[(valid['hips_fabs'] > 0) & (valid['ftir_ec'] > 0)]

        # Remove known Delhi outlier (2024-06-21, FTIR EC=60.6 — 5x the next highest)
        if site_name == 'Delhi':
            before_n = len(valid)
            valid = valid[valid['ftir_ec'] < 30]
            if len(valid) < before_n:
                print(f'  Delhi: removed {before_n - len(valid)} outlier(s) with FTIR EC > 30')

        if len(valid) >= 10:
            site_data[site_name] = valid
            print(f'{site_name}: {len(valid)} samples, '
                  f'HIPS BC [{valid["hips_fabs"].min():.2f}–{valid["hips_fabs"].max():.2f}], '
                  f'FTIR EC [{valid["ftir_ec"].min():.2f}–{valid["ftir_ec"].max():.2f}]')

print(f'\nSites loaded: {list(site_data.keys())}')

---

## Section 1: Cross-Site Comparison

Compare the HIPS floor and slope across all 4 sites to determine if the problem is Addis-specific.

---

In [ ]:
# =============================================================================
# 1: Cross-site scatter plots — HIPS BC vs FTIR EC
# =============================================================================

fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))

site_stats = []

for idx, (site_name, df) in enumerate(site_data.items()):
    ax = axes[idx]
    color = SITE_COLORS[site_name]
    x, y = df['ftir_ec'].values, df['hips_fabs'].values

    ax.scatter(x, y, s=40, alpha=0.7, c=color, edgecolors='black', linewidths=0.3)

    # OLS
    sl, intc, r, p, _ = stats.linregress(x, y)
    lim = max(x.max(), y.max()) * 1.1
    x_line = np.linspace(0, lim, 100)
    ax.plot(x_line, sl * x_line + intc, 'r-', linewidth=2, label='OLS')
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='1:1')

    # Floor line
    ax.axhline(y=y.min(), color=color, linestyle=':', linewidth=1.5, alpha=0.7,
               label=f'Floor: {y.min():.2f}')

    sign = '+' if intc >= 0 else '-'
    ax.text(0.05, 0.95,
            f'slope = {sl:.3f}\n'
            f'intercept = {intc:.3f}\n'
            f'R\u00b2 = {r**2:.3f}\n'
            f'n = {len(df)}\n'
            f'floor = {y.min():.2f}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Through origin
    sl_o = np.sum(x * y) / np.sum(x**2)

    site_stats.append({
        'Site': site_name, 'n': len(df),
        'HIPS_floor': y.min(), 'HIPS_max': y.max(),
        'FTIR_min': x.min(), 'FTIR_max': x.max(),
        'OLS_slope': sl, 'OLS_intercept': intc, 'r': r,
        'origin_slope': sl_o,
        'Fabs_floor': y.min() * MAC_VALUE,
    })

    ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
    ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)' if idx == 0 else '', fontsize=FONT['axis'])
    ax.set_title(site_name.replace('_', ' '), fontsize=FONT['title'],
                 fontweight='bold', color=color)
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.legend(fontsize=FONT['legend'] - 2, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('HIPS BC vs FTIR EC Across All Sites\n'
             '(dotted line = HIPS floor; red = OLS fit; dashed = 1:1)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(str(plots_dir / 'cross_site_comparison.png'), dpi=200, bbox_inches='tight')
plt.show()

# Summary table
stats_df = pd.DataFrame(site_stats)
stats_df['R\u00b2'] = stats_df['r'] ** 2
print('\n' + '=' * 90)
print('CROSS-SITE SUMMARY')
print('=' * 90)
print(stats_df[['Site', 'n', 'HIPS_floor', 'Fabs_floor', 'OLS_slope', 'OLS_intercept',
                'origin_slope', 'R\u00b2']].to_string(index=False))

---

## Section 2: Baseline Subtraction — Fixing Problem 1

Subtract the Fabs baseline from each site and show that it removes the intercept but leaves the slope unchanged.

---

In [ ]:
# =============================================================================
# 2: Baseline subtraction — before vs after for each site
# =============================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 11))

for idx, (site_name, df) in enumerate(site_data.items()):
    color = SITE_COLORS[site_name]
    x = df['ftir_ec'].values
    y_orig = df['hips_fabs'].values

    # Compute site-specific baseline from regression intercept
    sl_orig, intc_orig, r_orig, _, _ = stats.linregress(x, y_orig)
    baseline = intc_orig  # in BC units (Fabs/MAC)

    y_corr = y_orig - baseline

    # Top row: original
    ax = axes[0, idx]
    ax.scatter(x, y_orig, s=35, alpha=0.7, c=color, edgecolors='black', linewidths=0.3)
    lim = max(x.max(), y_orig.max()) * 1.1
    x_line = np.linspace(0, lim, 100)
    ax.plot(x_line, sl_orig * x_line + intc_orig, 'r-', linewidth=2)
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4)
    ax.text(0.05, 0.95,
            f'slope = {sl_orig:.3f}\nintercept = {intc_orig:.3f}\nR\u00b2 = {r_orig**2:.3f}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_title(f'{site_name.replace("_", " ")} \u2014 Original',
                 fontsize=FONT['title'] - 1, fontweight='bold', color=color)
    ax.set_xlabel('FTIR EC' if idx == 0 else '', fontsize=FONT['axis'] - 1)
    ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)', fontsize=FONT['axis'] - 1)
    ax.set_xlim(0, None)
    ax.set_ylim(0 if y_orig.min() > -0.5 else None, None)
    ax.grid(True, alpha=0.3)

    # Bottom row: baseline-subtracted
    ax = axes[1, idx]
    sl_corr, intc_corr, r_corr, _, _ = stats.linregress(x, y_corr)
    ax.scatter(x, y_corr, s=35, alpha=0.7, c=color, edgecolors='black', linewidths=0.3)
    lim2 = max(x.max(), max(y_corr.max(), 0)) * 1.1
    x_line = np.linspace(0, lim2, 100)
    ax.plot(x_line, sl_corr * x_line + intc_corr, 'r-', linewidth=2)
    ax.plot([0, lim2], [0, lim2], 'k--', alpha=0.4)
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.text(0.05, 0.95,
            f'slope = {sl_corr:.3f}  (unchanged!)\n'
            f'intercept = {intc_corr:.4f}  (\u2248 0)\n'
            f'R\u00b2 = {r_corr**2:.3f}\n'
            f'baseline removed: {baseline:.2f}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax.set_title(f'{site_name.replace("_", " ")} \u2014 Baseline Subtracted',
                 fontsize=FONT['title'] - 1, fontweight='bold', color=color)
    ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'] - 1)
    ax.set_ylabel('HIPS BC \u2212 baseline' if idx == 0 else '', fontsize=FONT['axis'] - 1)
    ax.set_xlim(0, None)
    ax.grid(True, alpha=0.3)

plt.suptitle('Baseline Subtraction Fixes the Intercept but NOT the Slope\n'
             '(top: original, bottom: after removing site-specific baseline)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'baseline_subtraction_comparison.png'),
            dpi=200, bbox_inches='tight')
plt.show()

---

## Section 3: Filter Loading / Saturation Hypothesis

Why is the slope only 0.40? Hypothesis: at high filter loading (high EC), the filter becomes so optically dense that HIPS **saturates** — adding more BC doesn't change absorption proportionally.

If true, the slope should decrease at higher concentrations, and sites with higher loadings should have lower slopes.

---

In [ ]:
# =============================================================================
# 3A: Does the slope depend on concentration level?
# Sliding-window regression for Addis Ababa
# =============================================================================

df_aa = site_data['Addis_Ababa'].copy()
df_aa_sorted = df_aa.sort_values('ftir_ec')

window_size = 40  # samples per window
step = 5
window_results = []

for start in range(0, len(df_aa_sorted) - window_size, step):
    window = df_aa_sorted.iloc[start:start + window_size]
    x_w, y_w = window['ftir_ec'].values, window['hips_fabs'].values
    if len(np.unique(x_w)) > 3:
        sl, intc, r, p, _ = stats.linregress(x_w, y_w)
        window_results.append({
            'ec_center': x_w.mean(),
            'ec_min': x_w.min(),
            'ec_max': x_w.max(),
            'slope': sl, 'intercept': intc, 'r': r, 'n': len(window),
        })

win_df = pd.DataFrame(window_results)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Slope vs EC concentration center
ax = axes[0]
ax.plot(win_df['ec_center'], win_df['slope'], 'o-', color='#E74C3C',
        markersize=5, linewidth=1.5)
ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, label='Ideal slope = 1.0')
ax.axhline(y=0.40, color='gray', linestyle=':', alpha=0.5, label='Overall slope = 0.40')
ax.fill_between(win_df['ec_center'], 0.9, 1.1, color='green', alpha=0.1)

r_slope, p_slope = stats.pearsonr(win_df['ec_center'], win_df['slope'])
ax.text(0.05, 0.05,
        f'r(EC center, slope) = {r_slope:.3f}\np = {p_slope:.4f}\n'
        f'Trend: slope {"DECREASES" if r_slope < 0 else "INCREASES"} with EC',
        transform=ax.transAxes, va='bottom', fontsize=FONT['annot'],
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('FTIR EC Window Center (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('Local Regression Slope', fontsize=FONT['axis'])
ax.set_title('Does Slope Depend on EC Level?\n(sliding window, n=40)',
             fontsize=FONT['title'], fontweight='bold')
ax.legend(fontsize=FONT['legend'])
ax.grid(True, alpha=0.3)

# Panel 2: Cross-site — slope vs mean EC
ax = axes[1]
for site_name, df in site_data.items():
    sl, intc, r, _, _ = stats.linregress(df['ftir_ec'], df['hips_fabs'])
    ax.scatter(df['ftir_ec'].mean(), sl, s=150, c=SITE_COLORS[site_name],
              marker=SITE_MARKERS[site_name], edgecolors='black', linewidths=1,
              label=f'{site_name.replace("_", " ")} (n={len(df)})', zorder=5)

# Fit trend across sites
means = [df['ftir_ec'].mean() for df in site_data.values()]
slopes = [stats.linregress(df['ftir_ec'], df['hips_fabs'])[0] for df in site_data.values()]
if len(means) >= 3:
    r_cross, p_cross = stats.pearsonr(means, slopes)
    ax.text(0.05, 0.05,
            f'r(mean EC, slope) = {r_cross:.3f}\n'
            f'Higher loading \u2192 {"lower" if r_cross < 0 else "higher"} slope',
            transform=ax.transAxes, va='bottom', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, alpha=0.5)
ax.set_xlabel('Mean FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('OLS Slope (HIPS BC vs FTIR EC)', fontsize=FONT['axis'])
ax.set_title('Cross-Site: Slope vs Loading Level',
             fontsize=FONT['title'], fontweight='bold')
ax.legend(fontsize=FONT['legend'] - 1)
ax.grid(True, alpha=0.3)

# Panel 3: Cross-site — slope vs HIPS floor
ax = axes[2]
floors = [df['hips_fabs'].min() for df in site_data.values()]
for site_name, sl_val, floor_val in zip(site_data.keys(), slopes, floors):
    ax.scatter(floor_val, sl_val, s=150, c=SITE_COLORS[site_name],
              marker=SITE_MARKERS[site_name], edgecolors='black', linewidths=1,
              label=site_name.replace('_', ' '), zorder=5)

if len(floors) >= 3:
    r_floor, p_floor = stats.pearsonr(floors, slopes)
    ax.text(0.05, 0.05,
            f'r(floor, slope) = {r_floor:.3f}\n'
            f'Higher floor \u2192 {"lower" if r_floor < 0 else "higher"} slope',
            transform=ax.transAxes, va='bottom', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, alpha=0.5)
ax.set_xlabel('HIPS BC Floor (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
ax.set_ylabel('OLS Slope', fontsize=FONT['axis'])
ax.set_title('Cross-Site: Slope vs HIPS Floor',
             fontsize=FONT['title'], fontweight='bold')
ax.legend(fontsize=FONT['legend'] - 1)
ax.grid(True, alpha=0.3)

plt.suptitle('Is the Low Slope Caused by Filter Loading / Optical Saturation?',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'loading_saturation_analysis.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# 3B: Non-linearity test — HIPS response curve
# Is the relationship actually linear, or does HIPS saturate at high EC?
# =============================================================================

fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))

for idx, (site_name, df) in enumerate(site_data.items()):
    ax = axes[idx]
    color = SITE_COLORS[site_name]
    x, y = df['ftir_ec'].values, df['hips_fabs'].values

    ax.scatter(x, y, s=35, alpha=0.6, c=color, edgecolors='black', linewidths=0.3)

    # Linear fit
    sl, intc, r_lin, _, _ = stats.linregress(x, y)
    x_line = np.linspace(0, x.max() * 1.05, 100)
    ax.plot(x_line, sl * x_line + intc, 'r-', linewidth=2, label=f'Linear (R\u00b2={r_lin**2:.3f})')

    # Quadratic fit — test for curvature
    if len(x) >= 15:
        coeffs = np.polyfit(x, y, 2)
        y_quad = np.polyval(coeffs, x_line)
        ss_res_lin = np.sum((y - (sl * x + intc))**2)
        ss_res_quad = np.sum((y - np.polyval(coeffs, x))**2)
        r2_lin = 1 - ss_res_lin / np.sum((y - y.mean())**2)
        r2_quad = 1 - ss_res_quad / np.sum((y - y.mean())**2)
        ax.plot(x_line, y_quad, 'b--', linewidth=1.5,
                label=f'Quadratic (R\u00b2={r2_quad:.3f})')

        # Curvature coefficient
        ax.text(0.05, 0.55,
                f'a\u00b7x\u00b2 coeff: {coeffs[0]:.4f}\n'
                f'\u0394R\u00b2 = {r2_quad - r2_lin:.4f}\n'
                f'{"SATURATING" if coeffs[0] < 0 else "ACCELERATING"}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'] - 1,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    ax.plot([0, x.max() * 1.1], [0, x.max() * 1.1], 'k--', alpha=0.3)
    ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'] - 1)
    ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)' if idx == 0 else '', fontsize=FONT['axis'] - 1)
    ax.set_title(site_name.replace('_', ' '), fontsize=FONT['title'] - 1,
                 fontweight='bold', color=color)
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.legend(fontsize=FONT['legend'] - 2)
    ax.grid(True, alpha=0.3)

plt.suptitle('Non-Linearity Test: Does HIPS Saturate at High Loading?\n'
             '(negative quadratic coefficient = saturation/compression)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(str(plots_dir / 'nonlinearity_test.png'), dpi=200, bbox_inches='tight')
plt.show()

---

## Section 4: Two-Parameter Correction

Fix both problems simultaneously:
- **Problem 1 (baseline)**: subtract site-specific Fabs baseline
- **Problem 2 (sensitivity)**: use site-specific effective MAC

Formula: `EC_corrected = (Fabs - baseline) / MAC_effective`

---

In [ ]:
# =============================================================================
# 4: Two-parameter correction for each site
# =============================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 11))

correction_params = []

for idx, (site_name, df) in enumerate(site_data.items()):
    color = SITE_COLORS[site_name]
    x = df['ftir_ec'].values
    y_fabs = df['hips_fabs'].values * MAC_VALUE  # Back to Fabs in Mm^-1

    # The regression Fabs = slope_fabs * FTIR_EC + intercept_fabs gives us:
    # intercept_fabs = Fabs baseline
    # slope_fabs = effective MAC
    sl_fabs, intc_fabs, r_orig, _, _ = stats.linregress(x, y_fabs)
    baseline_fabs = intc_fabs
    mac_effective = sl_fabs

    # Corrected BC = (Fabs - baseline) / MAC_effective
    y_corrected = (y_fabs - baseline_fabs) / mac_effective

    # Verify: should have slope \u2248 1, intercept \u2248 0
    sl_corr, intc_corr, r_corr, _, _ = stats.linregress(x, y_corrected)

    correction_params.append({
        'Site': site_name,
        'baseline_Fabs': baseline_fabs,
        'MAC_effective': mac_effective,
        'MAC_standard': MAC_VALUE,
        'slope_before': sl_fabs / MAC_VALUE,  # original slope in BC units
        'intc_before': intc_fabs / MAC_VALUE,
        'slope_after': sl_corr,
        'intc_after': intc_corr,
        'r': r_corr,
    })

    # Top row: original
    ax = axes[0, idx]
    y_orig_bc = y_fabs / MAC_VALUE
    ax.scatter(x, y_orig_bc, s=35, alpha=0.7, c=color, edgecolors='black', linewidths=0.3)
    lim = max(x.max(), y_orig_bc.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4)
    sl_o, intc_o = sl_fabs / MAC_VALUE, intc_fabs / MAC_VALUE
    x_line = np.linspace(0, lim, 100)
    ax.plot(x_line, sl_o * x_line + intc_o, 'r-', linewidth=2)
    ax.text(0.05, 0.95,
            f'slope = {sl_o:.3f}\nint = {intc_o:.3f}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_title(f'{site_name.replace("_", " ")} \u2014 Original (MAC={MAC_VALUE})',
                 fontsize=FONT['title'] - 2, fontweight='bold', color=color)
    ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)' if idx == 0 else '', fontsize=FONT['axis'] - 1)
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)

    # Bottom row: corrected
    ax = axes[1, idx]
    ax.scatter(x, y_corrected, s=35, alpha=0.7, c=color, edgecolors='black', linewidths=0.3)
    lim2 = max(x.max(), y_corrected.max()) * 1.1
    ax.plot([0, lim2], [0, lim2], 'k--', alpha=0.4)
    x_line = np.linspace(0, lim2, 100)
    ax.plot(x_line, sl_corr * x_line + intc_corr, 'g-', linewidth=2)
    ax.text(0.05, 0.95,
            f'slope = {sl_corr:.3f}  \u2713\n'
            f'int = {intc_corr:.4f}  \u2713\n'
            f'baseline = {baseline_fabs:.1f} Mm\u207b\u00b9\n'
            f'MAC_eff = {mac_effective:.1f}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    ax.set_title(f'{site_name.replace("_", " ")} \u2014 Corrected',
                 fontsize=FONT['title'] - 2, fontweight='bold', color=color)
    ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)', fontsize=FONT['axis'] - 1)
    ax.set_ylabel('Corrected HIPS BC' if idx == 0 else '', fontsize=FONT['axis'] - 1)
    ax.set_xlim(0, None)
    ax.grid(True, alpha=0.3)

plt.suptitle('Two-Parameter Correction: (Fabs \u2212 baseline) / MAC_effective\n'
             'Top: original (slope \u226a 1) | Bottom: corrected (slope \u2248 1)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'two_parameter_correction.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Correction parameters table
corr_tbl = pd.DataFrame(correction_params)
print('\n' + '=' * 90)
print('TWO-PARAMETER CORRECTION: Site-Specific Values')
print('=' * 90)
print(f'{"Site":<15s} {"Fabs baseline":>14s} {"MAC_eff":>8s} {"MAC_std":>8s} '
      f'{"slope before":>12s} {"slope after":>12s} {"int after":>10s}')
print('-' * 85)
for _, row in corr_tbl.iterrows():
    print(f'{row["Site"]:<15s} {row["baseline_Fabs"]:14.1f} {row["MAC_effective"]:8.1f} '
          f'{row["MAC_standard"]:8.0f} {row["slope_before"]:12.3f} '
          f'{row["slope_after"]:12.3f} {row["intc_after"]:10.4f}')

print(f'\nInterpretation: EC = (Fabs - baseline) / MAC_effective')
print(f'  Addis Ababa: EC = (Fabs - {corr_tbl.loc[corr_tbl["Site"]=="Addis_Ababa", "baseline_Fabs"].values[0]:.0f}) / '
      f'{corr_tbl.loc[corr_tbl["Site"]=="Addis_Ababa", "MAC_effective"].values[0]:.1f}')

---

## Section 5: Validation — Does Corrected HIPS Agree Better with Aethalometer?

If the two-parameter correction is real (not overfitting to FTIR EC), then corrected HIPS should also agree better with the independent Aethalometer IR BCc.

---

In [ ]:
# =============================================================================
# 5: Validation — corrected HIPS vs Aethalometer IR BCc
# =============================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 11))

for idx, (site_name, df) in enumerate(site_data.items()):
    color = SITE_COLORS[site_name]

    # Check if aethalometer data exists
    if 'ir_bcc' not in df.columns:
        for row_idx in range(2):
            axes[row_idx, idx].text(0.5, 0.5, 'No Aeth data',
                                    transform=axes[row_idx, idx].transAxes, ha='center')
            axes[row_idx, idx].set_title(site_name.replace('_', ' '))
        continue

    valid = df.dropna(subset=['hips_fabs', 'ir_bcc']).copy()
    valid = valid[(valid['hips_fabs'] > 0) & (valid['ir_bcc'] > 0)]

    if len(valid) < 10:
        for row_idx in range(2):
            axes[row_idx, idx].text(0.5, 0.5, f'n={len(valid)}',
                                    transform=axes[row_idx, idx].transAxes, ha='center')
        continue

    y_fabs = valid['hips_fabs'].values * MAC_VALUE
    y_aeth = valid['ir_bcc'].values

    # Get correction params for this site
    site_corr = corr_tbl[corr_tbl['Site'] == site_name].iloc[0]
    baseline = site_corr['baseline_Fabs']
    mac_eff = site_corr['MAC_effective']

    y_orig = y_fabs / MAC_VALUE
    y_corrected = (y_fabs - baseline) / mac_eff

    # Top row: original HIPS BC vs Aeth
    ax = axes[0, idx]
    ax.scatter(y_aeth, y_orig, s=35, alpha=0.7, c=color, edgecolors='black', linewidths=0.3)
    sl_o, intc_o, r_o, _, _ = stats.linregress(y_aeth, y_orig)
    lim = max(y_aeth.max(), y_orig.max()) * 1.1
    x_line = np.linspace(0, lim, 100)
    ax.plot(x_line, sl_o * x_line + intc_o, 'r-', linewidth=2)
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4)
    ax.text(0.05, 0.95,
            f'slope = {sl_o:.3f}\nint = {intc_o:.3f}\nR\u00b2 = {r_o**2:.3f}\nn = {len(valid)}',
            transform=ax.transAxes, va='top', fontsize=FONT['annot'],
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_title(f'{site_name.replace("_", " ")} \u2014 Original',
                 fontsize=FONT['title'] - 2, fontweight='bold', color=color)
    ax.set_ylabel('HIPS BC (\u00b5g/m\u00b3)' if idx == 0 else '', fontsize=FONT['axis'] - 1)
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)

    # Bottom row: corrected HIPS vs Aeth
    ax = axes[1, idx]
    mask = y_corrected > 0
    ax.scatter(y_aeth[mask], y_corrected[mask], s=35, alpha=0.7, c=color,
              edgecolors='black', linewidths=0.3)
    if mask.sum() >= 5:
        sl_c, intc_c, r_c, _, _ = stats.linregress(y_aeth[mask], y_corrected[mask])
        lim2 = max(y_aeth[mask].max(), y_corrected[mask].max()) * 1.1
        x_line = np.linspace(0, lim2, 100)
        ax.plot(x_line, sl_c * x_line + intc_c, 'g-', linewidth=2)
        ax.plot([0, lim2], [0, lim2], 'k--', alpha=0.4)
        ax.text(0.05, 0.95,
                f'slope = {sl_c:.3f}\nint = {intc_c:.3f}\nR\u00b2 = {r_c**2:.3f}\n'
                f'\u0394slope = {sl_c - sl_o:+.3f}\n\u0394|int| = {abs(intc_c) - abs(intc_o):+.3f}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    ax.set_title(f'{site_name.replace("_", " ")} \u2014 Corrected',
                 fontsize=FONT['title'] - 2, fontweight='bold', color=color)
    ax.set_xlabel('Aeth IR BCc (\u00b5g/m\u00b3)', fontsize=FONT['axis'] - 1)
    ax.set_ylabel('Corrected HIPS BC' if idx == 0 else '', fontsize=FONT['axis'] - 1)
    ax.set_xlim(0, None)
    ax.grid(True, alpha=0.3)

plt.suptitle('Validation: Does Corrected HIPS Agree Better with Aethalometer?\n'
             'Top: original HIPS BC | Bottom: two-parameter corrected HIPS BC',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'validation_vs_aethalometer.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# Summary
# =============================================================================

print('=' * 80)
print('SUMMARY: HIPS TWO-PROBLEM DECOMPOSITION')
print('=' * 80)

print('\n1. PROBLEM 1 — ADDITIVE BASELINE')
for _, row in corr_tbl.iterrows():
    print(f'   {row["Site"]:<15s}: Fabs baseline = {row["baseline_Fabs"]:6.1f} Mm\u207b\u00b9 '
          f'(= {row["baseline_Fabs"]/MAC_VALUE:.2f} \u00b5g/m\u00b3 at MAC={MAC_VALUE})')

print('\n2. PROBLEM 2 — COMPRESSED SENSITIVITY')
for _, row in corr_tbl.iterrows():
    ratio = MAC_VALUE / row['MAC_effective'] if row['MAC_effective'] != 0 else np.nan
    print(f'   {row["Site"]:<15s}: MAC_eff = {row["MAC_effective"]:5.1f} '
          f'(standard MAC={MAC_VALUE} \u2192 slope only captures {ratio:.0%} of signal)')

print('\n3. CORRECTION FORMULA')
print('   EC_corrected = (Fabs - baseline) / MAC_effective')
print('   This is equivalent to a calibration transfer function.')

print('\n4. SITE COMPARISON')
for _, row in corr_tbl.iterrows():
    severity = 'MINIMAL' if abs(row['slope_before'] - 1) < 0.25 else \
               'MODERATE' if abs(row['slope_before'] - 1) < 0.5 else 'SEVERE'
    print(f'   {row["Site"]:<15s}: {severity} '
          f'(original slope = {row["slope_before"]:.3f}, floor = {row["baseline_Fabs"]/MAC_VALUE:.2f})')

print('\n' + '=' * 80)